In [0]:
from pyspark.sql import functions as F

silver = spark.table("ecom_external_catalog.retail_sales_dev.sales_silver")

gold_daily_sales = (
    silver
    .withColumn("sale_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("sale_date", "category")
    .agg(
        F.count("*").alias("sale_count"),
        F.sum("quantity").alias("units_sold"),
        F.sum("sale_revenue").alias("revenue"),
    )
)

display(gold_daily_sales.orderBy("sale_date", "category"))

In [0]:
gold_table = "ecom_external_catalog.retail_sales_dev.gold_daily_sales"

(
    gold_daily_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

In [0]:
from pyspark.sql import functions as F

saved_gold = spark.table(gold_table)

display(
    saved_gold.agg(
        F.count("*").alias("category_day_groups"),
        F.sum("sale_count").alias("total_sales"),
        F.sum("revenue").alias("total_revenue"),
    )
)
